In [ ]:
from __future__ import annotations

import operator
import os
import json
import re
from datetime import date
from pathlib import Path
from typing import TypedDict, List, Optional, Literal, Annotated

from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.tools.tavily_search import TavilySearchResults

from dotenv import load_dotenv
load_dotenv()



✅ Imports loaded successfully


In [ ]:


class SearchQuery(BaseModel):
    query: str = Field(..., description="A precise search query string (5–12 words)")
    domain: Literal[
        "academic_papers",
        "preprints",
        "patents_applied",
        "grey_literature",
        "news_commentary",
        "conference_proceedings"
    ] = Field(..., description="Target source domain for this query")
    rationale: str = Field(..., description="Why this query is important for the topic")


class TopicDecomposition(BaseModel):
    topic: str
    scope_statement: str = Field(..., description="1–2 sentence definition of the research scope")
    core_subtopics: List[str] = Field(..., min_length=4, max_length=10,
        description="Primary sub-fields and adjacent areas")
    disciplines: List[str] = Field(..., description="Academic disciplines involved")
    time_periods: List[str] = Field(..., description="Key historical periods and recent developments")
    search_queries: List[SearchQuery] = Field(..., min_length=10, max_length=20,
        description="10–20 precise queries across all source domains")



class EvidenceItem(BaseModel):
    title: str
    url: str
    snippet: Optional[str] = None
    source: Optional[str] = None
    domain: str = "unknown"
    query_used: str = ""


class SearchTask(BaseModel):
    query: str
    domain: str
    rationale: str



class EvidenceEvaluation(BaseModel):
    consensus_areas: List[str] = Field(...,
        description="Topics where strong empirical consensus exists")
    contested_claims: List[str] = Field(...,
        description="Findings that are debated or contested in the literature")
    key_debates: List[str] = Field(...,
        description="Fundamental unresolved disagreements between researchers")
    replication_concerns: List[str] = Field(default_factory=list,
        description="Results with known replication issues or null contradictions")
    methodological_landscape: str = Field(...,
        description="Summary of dominant methods, strengths, limitations, and emerging approaches")
    quality_notes: str = Field(...,
        description="Overall quality assessment of the evidence corpus")



class SynthesisTask(BaseModel):
    output_type: Literal["literature_summary", "knowledge_map", "annotated_bibliography"]
    topic: str
    decomposition_json: str  # serialized TopicDecomposition
    evidence_json: str       # serialized list of EvidenceItem
    evaluation_json: str     # serialized EvidenceEvaluation


print("✅ All schemas defined")

✅ All schemas defined


## 2. LangGraph State

In [4]:
class ResearchState(TypedDict):
    # Input
    topic: str
    scope: Optional[str]
    focus: Optional[str]

    # Phase 1
    decomposition: TopicDecomposition

    # Phase 2 — parallel evidence accumulation via operator.add
    evidence_items: Annotated[List[EvidenceItem], operator.add]

    # Phase 3
    evaluation: EvidenceEvaluation

    # Phase 4 — parallel synthesis outputs via operator.add
    synthesis_outputs: Annotated[List[str], operator.add]   # [literature_md, knowledge_map_md, bibliography_md]

    # Final assembled report
    final_report: str
    output_path: str


print("✅ ResearchState defined")

✅ ResearchState defined


## 3. LLM & Tools

In [5]:
# GPT-4.1 for deep synthesis tasks
llm_strong = ChatOpenAI(model="gpt-4.1", temperature=0.2)

# GPT-4.1-mini for lighter orchestration tasks
llm_fast = ChatOpenAI(model="gpt-4.1-mini", temperature=0.1)

# Tavily web search (covers academic, preprint, news, grey literature)
search_tool = TavilySearchResults(
    max_results=8,
    search_depth="advanced",
    include_answer=True,
    include_raw_content=False,
    include_images=False,
)

print("✅ LLMs and search tool initialized")

✅ LLMs and search tool initialized


C:\Users\lamic\AppData\Local\Temp\ipykernel_11484\1701091435.py:8: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(


## 4. Node Functions

In [ ]:


def topic_decomposer(state: ResearchState) -> dict:
    """Breaks the topic into subtopics and generates 10–20 targeted search queries."""
    topic = state["topic"]
    scope = state.get("scope") or "Exhaustive global coverage, all time periods"
    focus = state.get("focus") or "No specific focus — cover all major angles"

    print(f"\n🔬 Phase 1 — Decomposing topic: '{topic}'")

    decomposition: TopicDecomposition = llm_fast.with_structured_output(TopicDecomposition).invoke([
        SystemMessage(content="""You are an elite academic research strategist.
Your task is to decompose a research topic into a systematic search plan for a PhD-level literature review.

You must:
1. Define the scope clearly in 1–2 sentences
2. Identify 4–10 core subtopics and adjacent research areas
3. List all relevant academic disciplines
4. Identify key time periods in the field's development
5. Generate 10–20 precise search queries covering ALL source domains:
   - academic_papers (peer-reviewed journals, systematic reviews, meta-analyses)
   - preprints (arXiv, bioRxiv, SSRN, medRxiv)
   - patents_applied (USPTO, EPO — for technical/applied topics)
   - grey_literature (technical reports, white papers, dissertations, policy docs)
   - news_commentary (science journalism, expert blogs, institutional press releases)
   - conference_proceedings (major venues in the field)

Queries must be specific, targeted, and varied — not generic reformulations of the same idea.
Ensure coverage of: foundational works, recent breakthroughs, debates/controversies, applied research."""),
        HumanMessage(content=f"""TOPIC: {topic}
SCOPE: {scope}
FOCUS: {focus}

Produce the complete topic decomposition and search plan.""")
    ])

    print(f"   ✓ Identified {len(decomposition.core_subtopics)} subtopics")
    print(f"   ✓ Generated {len(decomposition.search_queries)} search queries")
    print(f"   ✓ Disciplines: {', '.join(decomposition.disciplines[:4])}...")

    return {"decomposition": decomposition}


print("✅ topic_decomposer defined")

✅ topic_decomposer defined


In [ ]:


def route_search_tasks(state: ResearchState) -> List[Send]:
    """Fan-out: create one Send per search query for parallel execution."""
    decomposition = state["decomposition"]
    print(f"\n🔀 Routing {len(decomposition.search_queries)} parallel search tasks...")
    return [
        Send(
            "search_worker",
            {
                "topic": state["topic"],
                "query": sq.query,
                "domain": sq.domain,
                "rationale": sq.rationale,
            }
        )
        for sq in decomposition.search_queries
    ]


print("✅ route_search_tasks defined")

✅ route_search_tasks defined


In [ ]:


class SearchWorkerState(TypedDict):
    topic: str
    query: str
    domain: str
    rationale: str
    evidence_items: Annotated[List[EvidenceItem], operator.add]  # fans back into main state


def search_worker(state: SearchWorkerState) -> dict:
    """Executes a single search query and returns structured evidence items."""
    query = state["query"]
    domain = state["domain"]
    topic = state["topic"]

    print(f"   🔍 [{domain}] Searching: '{query[:60]}...'" if len(query) > 60 else f"   🔍 [{domain}] Searching: '{query}'")

    # Augment query with domain-specific context for better results
    domain_prefixes = {
        "academic_papers": f"site:scholar.google.com OR site:pubmed.ncbi.nlm.nih.gov OR site:semanticscholar.org {query}",
        "preprints": f"site:arxiv.org OR site:biorxiv.org OR site:ssrn.com {query}",
        "patents_applied": f"patent OR USPTO OR EPO {query} application",
        "grey_literature": f"technical report OR white paper OR dissertation OR policy {query}",
        "news_commentary": f"research news OR expert analysis {query} 2023 OR 2024 OR 2025",
        "conference_proceedings": f"conference proceedings OR symposium paper {query}",
    }
    augmented_query = domain_prefixes.get(domain, query)

    try:
        raw_results = search_tool.invoke(augmented_query)
    except Exception as e:
        print(f"   ⚠️  Search failed for '{query}': {e}")
        return {"evidence_items": []}

    items: List[EvidenceItem] = []
    for r in raw_results:
        if isinstance(r, dict):
            items.append(EvidenceItem(
                title=r.get("title", "Untitled"),
                url=r.get("url", ""),
                snippet=r.get("content", "")[:500] if r.get("content") else r.get("snippet", "")[:500],
                source=r.get("score", ""),
                domain=domain,
                query_used=query
            ))

    print(f"      → Found {len(items)} results")
    return {"evidence_items": items}


print("✅ search_worker defined")

✅ search_worker defined


In [ ]:


def evidence_aggregator(state: ResearchState) -> dict:
    """Deduplicates and summarizes the collected evidence corpus."""
    items = state.get("evidence_items", [])

    # Deduplicate by URL
    seen_urls = set()
    unique_items = []
    for item in items:
        if item.url not in seen_urls and item.url:
            seen_urls.add(item.url)
            unique_items.append(item)

    print(f"\n📚 Phase 2 Complete — Evidence aggregated")
    print(f"   ✓ Total results: {len(items)}")
    print(f"   ✓ Unique sources: {len(unique_items)}")

    domain_counts = {}
    for item in unique_items:
        domain_counts[item.domain] = domain_counts.get(item.domain, 0) + 1
    for d, c in domain_counts.items():
        print(f"   ✓ {d}: {c} sources")

    return {"evidence_items": unique_items}


print("✅ evidence_aggregator defined")

✅ evidence_aggregator defined


In [ ]:

def critical_evaluator(state: ResearchState) -> dict:
    """Critically evaluates the evidence corpus: consensus, debates, methodology."""
    topic = state["topic"]
    items = state.get("evidence_items", [])
    decomposition = state["decomposition"]

    print(f"\n🔎 Phase 3 — Critical evaluation of {len(items)} sources...")

    # Prepare a compact evidence digest for the LLM
    evidence_digest = "\n\n".join([
        f"[{i+1}] DOMAIN: {item.domain}\nTITLE: {item.title}\nURL: {item.url}\nSNIPPET: {item.snippet or 'N/A'}"
        for i, item in enumerate(items[:60])  # cap at 60 to stay within context
    ])

    evaluation: EvidenceEvaluation = llm_strong.with_structured_output(EvidenceEvaluation).invoke([
        SystemMessage(content="""You are a senior academic peer-reviewer conducting critical evaluation of a research corpus.

Your task is to:
1. Identify areas of STRONG EMPIRICAL CONSENSUS — where findings consistently agree across sources
2. Flag CONTESTED CLAIMS — findings supported by some sources but contradicted or challenged by others
3. Map KEY DEBATES — fundamental unresolved theoretical or empirical disagreements between schools of thought
4. Note REPLICATION CONCERNS — findings with known reproducibility problems or null results in the literature
5. Describe the METHODOLOGICAL LANDSCAPE — dominant methods, their strengths/limitations, emerging approaches
6. Provide QUALITY NOTES — overall assessment of evidence quality, potential biases, gaps

Be specific. Cite titles when possible. Distinguish clearly between:
  - Established consensus (multiple independent replications)
  - Emerging evidence (promising but not yet consolidated)
  - Contested claims (active debate)
  - Speculative ideas (theoretical, limited empirical support)"""),
        HumanMessage(content=f"""RESEARCH TOPIC: {topic}

SUBTOPICS IDENTIFIED: {', '.join(decomposition.core_subtopics)}

EVIDENCE CORPUS ({len(items)} sources):
{evidence_digest}

Conduct rigorous critical evaluation of this evidence corpus.""")
    ])

    print(f"   ✓ Consensus areas identified: {len(evaluation.consensus_areas)}")
    print(f"   ✓ Contested claims flagged: {len(evaluation.contested_claims)}")
    print(f"   ✓ Key debates mapped: {len(evaluation.key_debates)}")

    return {"evaluation": evaluation}


print("✅ critical_evaluator defined")

✅ critical_evaluator defined


In [ ]:


def route_synthesis_tasks(state: ResearchState) -> List[Send]:
    """Fan-out: run all 3 output writers in parallel."""
    topic = state["topic"]
    decomposition_json = state["decomposition"].model_dump_json()
    evidence_json = json.dumps([e.model_dump() for e in state.get("evidence_items", [])])
    evaluation_json = state["evaluation"].model_dump_json()

    output_types = ["literature_summary", "knowledge_map", "annotated_bibliography"]
    print(f"\n🔀 Phase 4 — Routing {len(output_types)} parallel synthesis tasks...")

    return [
        Send(
            "synthesis_worker",
            {
                "output_type": ot,
                "topic": topic,
                "decomposition_json": decomposition_json,
                "evidence_json": evidence_json,
                "evaluation_json": evaluation_json,
            }
        )
        for ot in output_types
    ]


print("✅ route_synthesis_tasks defined")

✅ route_synthesis_tasks defined


In [ ]:


class SynthesisWorkerState(TypedDict):
    output_type: str
    topic: str
    decomposition_json: str
    evidence_json: str
    evaluation_json: str
    synthesis_outputs: Annotated[List[str], operator.add]


SYNTHESIS_PROMPTS = {

"literature_summary": """You are an elite academic writer producing OUTPUT 1 — Literature Summary for a PhD-level research report.

Produce a comprehensive, narrative synthesis organized EXACTLY as follows:

## OUTPUT 1 — Literature Summary

### 1.1 Field Overview
- Define the topic and its scope
- Situate it within broader academic discourse
- State the current state of knowledge (2 substantial paragraphs)

### 1.2 Historical Development
- Trace the intellectual lineage
- Identify founding works, paradigm shifts, inflection points
- How has the field evolved over time?

### 1.3 Core Themes & Findings
- Organize by THEME (not by paper)
- For each theme: consensus, key evidence, open questions
- Minimum 4–6 distinct themes

### 1.4 Methodological Landscape
- Dominant methods and their strengths/limitations
- Emerging methodologies

### 1.5 Debates, Contradictions & Controversies
- Fundamental researcher disagreements
- Replication failures
- Most contentious open questions

### 1.6 Applied & Commercial Landscape
- Patent trends and applied research directions
- Grey literature insights
- Gap between academic research and real-world application

### 1.7 Research Gaps & Future Directions
- Unanswered questions
- Where is the field heading?
- Required methodological/conceptual advances

STANDARDS:
- Write in precise, formal academic English for a PhD audience
- Use inline citation notation (Author, Year) where possible
- Distinguish clearly: established consensus / emerging evidence / contested claims / speculative ideas
- NEVER fabricate citations — flag uncertain details with [VERIFY]
- Minimum 1500 words. Be thorough, not superficial.""",

"knowledge_map": """You are an elite academic cartographer producing OUTPUT 2 — Knowledge Map for a PhD-level research report.

Produce a structured, hierarchical breakdown organized EXACTLY as follows:

## OUTPUT 2 — Knowledge Map

### 2.1 Core Concepts Glossary
List and define the 15–30 most essential concepts. For each:
**Term:** [name]
**Definition:** [2–4 precise technical sentences]
**Origin:** [First introduced by / key associated work]
**Current Usage:** [how the term is used today; any semantic evolution]

### 2.2 Concept Relationship Map
Describe relationships between core concepts in this format:
[Concept A] → [relationship type] → [Concept B]
Use: builds on / contradicts / is a subset of / enables / is measured by / is debated alongside
Provide at least 15–20 relationships.

### 2.3 Key Research Questions
List 10–15 most important open questions, for each state:
- **Q:** [question]
- **Known:** [what is established]
- **Unknown:** [what remains unresolved]
- **Why it matters:** [scientific/practical significance]

### 2.4 Key Researchers & Institutions
List 10–20 most influential researchers:
**Name | Affiliation | Key Contribution | Landmark Work**
Also list top labs and research centers.

### 2.5 Landmark Papers & Milestones
Curated list organized as:
- Founding/seminal papers
- Most cited works
- Recent breakthroughs (last 3–5 years)
- Influential preprints
- Key conference papers
For each: **Title · Authors · Year · Venue · Why it matters (2–3 sentences)**

### 2.6 Datasets, Benchmarks & Tools
- Key datasets used in the field
- Standard benchmarks and evaluation frameworks
- Major software tools, libraries, experimental platforms

STANDARDS:
- Be specific and technically precise throughout
- Flag uncertain details with [VERIFY]
- Do not fabricate author names, paper titles, or institutions""",

"annotated_bibliography": """You are an elite academic librarian producing OUTPUT 3 — Annotated Bibliography for a PhD-level research report.

Produce a structured, critically annotated reference list organized EXACTLY as follows:

## OUTPUT 3 — Annotated Bibliography

For EACH entry use this exact format:
**[Author(s)] ([Year]). "[Title]." *Venue/Journal*, [details if known].**
- **Type:** [Peer-reviewed / Preprint / Patent / Conference / Grey literature / News]
- **Access:** [DOI / URL / arXiv ID / Patent number]
- **Summary:** 3–5 sentences on aims, methods, findings
- **Significance:** 2–3 sentences on why this work matters
- **Limitations:** 1–2 sentences on weaknesses

Organize entries into these SEVEN categories:

### Category 1: Foundational & Seminal Works
### Category 2: Recent High-Impact Papers (last 5 years)
### Category 3: Influential Preprints
### Category 4: Key Conference Papers
### Category 5: Patents & Applied Research
### Category 6: Grey Literature & Reports
### Category 7: Expert Commentary & Science Journalism

STANDARDS:
- Aim for 30–50 entries minimum; more for active fields
- Use only real sources from the evidence provided
- Flag uncertain details with [VERIFY]
- NEVER fabricate citations, DOIs, or author names
- If a source's category is unclear, place it in the best-fit category and note it"""
}


def synthesis_worker(state: SynthesisWorkerState) -> dict:
    """Generates one of the three PhD research outputs."""
    output_type = state["output_type"]
    topic = state["topic"]

    output_labels = {
        "literature_summary": "OUTPUT 1: Literature Summary",
        "knowledge_map": "OUTPUT 2: Knowledge Map",
        "annotated_bibliography": "OUTPUT 3: Annotated Bibliography"
    }
    print(f"   ✍️  Writing {output_labels[output_type]}...")

    system_prompt = SYNTHESIS_PROMPTS[output_type]

    # Parse and summarize evidence for context
    evidence_list = json.loads(state["evidence_json"])
    evaluation = json.loads(state["evaluation_json"])
    decomposition = json.loads(state["decomposition_json"])

    evidence_digest = "\n\n".join([
        f"[{i+1}] {e.get('domain','?').upper()} | {e.get('title','Untitled')}\nURL: {e.get('url','')}\n{e.get('snippet','')[:300]}"
        for i, e in enumerate(evidence_list[:80])
    ])

    user_message = f"""RESEARCH TOPIC: {topic}

SCOPE: {decomposition.get('scope_statement', '')}
SUBTOPICS: {', '.join(decomposition.get('core_subtopics', []))}
DISCIPLINES: {', '.join(decomposition.get('disciplines', []))}
TIME PERIODS: {', '.join(decomposition.get('time_periods', []))}

CRITICAL EVALUATION SUMMARY:
- Consensus Areas: {'; '.join(evaluation.get('consensus_areas', []))}
- Contested Claims: {'; '.join(evaluation.get('contested_claims', []))}
- Key Debates: {'; '.join(evaluation.get('key_debates', []))}
- Methodology Notes: {evaluation.get('methodological_landscape', '')}

EVIDENCE CORPUS ({len(evidence_list)} sources):
{evidence_digest}

Now produce the complete {output_labels[output_type]}."""

    result = llm_strong.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_message)
    ])

    output_text = result.content
    print(f"      → {output_labels[output_type]} written ({len(output_text.split())} words)")

    return {"synthesis_outputs": [output_text]}


print("✅ synthesis_worker defined")

✅ synthesis_worker defined


In [ ]:


def final_assembler(state: ResearchState) -> dict:
    """Assembles all three synthesis outputs into a single cohesive report."""
    topic = state["topic"]
    outputs = state.get("synthesis_outputs", [])
    today = date.today().strftime("%B %d, %Y")

    print(f"\n📎 Assembling final report from {len(outputs)} synthesis outputs...")

    # Identify which output is which
    lit_summary = ""
    knowledge_map = ""
    bibliography = ""

    for output in outputs:
        if "OUTPUT 1" in output or "Literature Summary" in output:
            lit_summary = output
        elif "OUTPUT 2" in output or "Knowledge Map" in output:
            knowledge_map = output
        elif "OUTPUT 3" in output or "Annotated Bibliography" in output:
            bibliography = output

    # Build the complete report
    header = f"""# PhD Research Report
## Topic: {topic}
**Generated:** {today}  
**Agent:** Advanced PhD Research Agent (LangGraph + GPT-4.1)  
**Evidence Corpus:** {len(state.get('evidence_items', []))} unique sources across {len(state['decomposition'].core_subtopics)} subtopics

---

> *This report was produced by an automated PhD-level research agent. All citations marked [VERIFY]
> should be independently verified before use in academic work.*

---

## Table of Contents
1. [Literature Summary](#output-1--literature-summary)
2. [Knowledge Map](#output-2--knowledge-map)
3. [Annotated Bibliography](#output-3--annotated-bibliography)

---
"""

    final_report = header + "\n\n" + lit_summary + "\n\n---\n\n" + knowledge_map + "\n\n---\n\n" + bibliography

    print(f"   ✓ Final report assembled: {len(final_report.split())} words")

    return {"final_report": final_report}


print("✅ final_assembler defined")

✅ final_assembler defined


In [ ]:


def save_outputs(state: ResearchState) -> dict:
    """Saves the final report as a Markdown file."""
    topic = state["topic"]
    report = state["final_report"]

    # Create safe filename
    safe_topic = re.sub(r'[^\w\s-]', '', topic.lower())
    safe_topic = re.sub(r'[\s]+', '_', safe_topic)[:60]
    today_str = date.today().strftime("%Y%m%d")
    filename = f"{today_str}_{safe_topic}_phd_research_report.md"

    # Save in the AI_Agents directory (same folder as this notebook)
    output_dir = Path(".")  # current directory
    output_path = output_dir / filename

    output_path.write_text(report, encoding="utf-8")

    print(f"\n💾 Report saved to: {output_path.resolve()}")
    print(f"   ✓ File size: {output_path.stat().st_size / 1024:.1f} KB")

    return {"output_path": str(output_path.resolve())}


print("✅ save_outputs defined")

✅ save_outputs defined


## 5. Build the LangGraph

In [ ]:


builder = StateGraph(ResearchState)

# ── Add nodes ──
builder.add_node("topic_decomposer",  topic_decomposer)
builder.add_node("search_worker",     search_worker)
builder.add_node("evidence_aggregator", evidence_aggregator)
builder.add_node("critical_evaluator",  critical_evaluator)
builder.add_node("synthesis_worker",    synthesis_worker)
builder.add_node("final_assembler",     final_assembler)
builder.add_node("save_outputs",        save_outputs)

# ── Edges ──

# Phase 1: START → topic_decomposer
builder.add_edge(START, "topic_decomposer")

# Phase 2: topic_decomposer → fan-out N search_workers (via Send)
builder.add_conditional_edges(
    "topic_decomposer",
    route_search_tasks,
    ["search_worker"]
)

# Phase 2 fan-in: all search_workers → evidence_aggregator
builder.add_edge("search_worker", "evidence_aggregator")

# Phase 3: evidence_aggregator → critical_evaluator
builder.add_edge("evidence_aggregator", "critical_evaluator")

# Phase 4: critical_evaluator → fan-out 3 synthesis_workers (via Send)
builder.add_conditional_edges(
    "critical_evaluator",
    route_synthesis_tasks,
    ["synthesis_worker"]
)

# Phase 4 fan-in: all synthesis_workers → final_assembler
builder.add_edge("synthesis_worker", "final_assembler")

# Final: assembler → save → END
builder.add_edge("final_assembler", "save_outputs")
builder.add_edge("save_outputs", END)

# Compile
graph = builder.compile()

print("✅ LangGraph compiled successfully!")
print()
print("Graph topology:")
print("  START")
print("    └─► topic_decomposer")
print("          └─► [N × search_worker] (parallel)")
print("                └─► evidence_aggregator")
print("                      └─► critical_evaluator")
print("                            └─► [3 × synthesis_worker] (parallel)")
print("                                  └─► final_assembler")
print("                                        └─► save_outputs")
print("                                              └─► END")

✅ LangGraph compiled successfully!

Graph topology:
  START
    └─► topic_decomposer
          └─► [N × search_worker] (parallel)
                └─► evidence_aggregator
                      └─► critical_evaluator
                            └─► [3 × synthesis_worker] (parallel)
                                  └─► final_assembler
                                        └─► save_outputs
                                              └─► END


In [ ]:

RESEARCH_INPUT = {
    "topic": "Large Language Models in Scientific Discovery",

    # Optional: constrain time period, geography, sub-domain
    "scope": "Focus on peer-reviewed work from 2020–2025, emphasis on natural sciences and biomedicine",

    # Optional: specific angle to emphasize
    "focus": "Emphasize empirical benchmarks, reproducibility concerns, and gaps between claimed and real-world utility",

    # Annotated state defaults (LangGraph requires these)
    "evidence_items": [],
    "synthesis_outputs": [],
}

print(f"🎓 Starting PhD Research Agent")
print(f"   Topic  : {RESEARCH_INPUT['topic']}")
print(f"   Scope  : {RESEARCH_INPUT['scope']}")
print(f"   Focus  : {RESEARCH_INPUT['focus']}")
print(f"   Time   : {date.today().strftime('%B %d, %Y')}")
print("=" * 70)

# ── Run the agent ──
# 
final_state = graph.invoke(
    RESEARCH_INPUT,
    config={"recursion_limit": 200}
)

print("\n" + "=" * 70)
print("✅ RESEARCH COMPLETE")
print(f"   Report saved to: {final_state['output_path']}")
print(f"   Evidence sources: {len(final_state['evidence_items'])}")
print(f"   Report length: {len(final_state['final_report'].split())} words")

🎓 Starting PhD Research Agent
   Topic  : Large Language Models in Scientific Discovery
   Scope  : Focus on peer-reviewed work from 2020–2025, emphasis on natural sciences and biomedicine
   Focus  : Emphasize empirical benchmarks, reproducibility concerns, and gaps between claimed and real-world utility
   Time   : April 12, 2026

🔬 Phase 1 — Decomposing topic: 'Large Language Models in Scientific Discovery'
   ✓ Identified 10 subtopics
   ✓ Generated 19 search queries
   ✓ Disciplines: Computer Science, Artificial Intelligence, Machine Learning, Natural Language Processing...

🔀 Routing 19 parallel search tasks...
   🔍 [academic_papers] Searching: 'large language models scientific discovery benchmarks 2020.....'
   🔍 [academic_papers] Searching: 'LLM reproducibility challenges biomedical applications'
   🔍 [academic_papers] Searching: 'transformer models drug discovery empirical evaluation'
   🔍 [preprints] Searching: 'large language models scientific discovery preprint 2020..20...'

ValidationError: 1 validation error for EvidenceItem
source
  Input should be a valid string [type=string_type, input_value=0.7742597, input_type=float]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type